# Esperienza 6: diffrazione da una fenditura

Fit: $y = A\left[\dfrac{\sin(B(x-C))}{B(x-C)}\right]^2 + D$ 

per cambare laser basta cambiare `nome`, `lam_nm` e lo sheet Excel 

In [43]:
import ROOT
import numpy as np
import pandas as pd

ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati
ROOT.gStyle.SetFitFormat(
    "4.2e"
)  # con 2 cifre decimali

In [44]:
#fit
def sinc2(x, par):
    t = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    D = par[3]
    u = B*(t - C)
    return A*np.sinc(u/np.pi)**2 + D      # np.sinc(z) = sin(pi z)/(pi z), quindi qui è sin(u)/u

def modellopy(t, A, B, C, D):
    u = B*(t - C)
    return A*np.sinc(u/np.pi)**2 + D

def sinc2_deriv(x, par):
    t = x
    dt = 1e-6
    return ((modellopy(t+dt/2, par[0], par[1], par[2], par[3])-modellopy(t-dt/2, par[0], par[1], par[2], par[3]))/dt)

In [45]:
nome = "rosso"        #  per nomi  immagini
lam_nm = 650.0        # lunghezza d'onda nm
s = 2500.0            # distanza fenditura-fotodiodo  mm
err_s = 10.0          # incertezza su s  mm

xlsx = "Esperienza6_simulata.xlsx"
df = pd.read_excel(xlsx, sheet_name="Dati")           # V = segnale - fondo
df = df.dropna(subset=["x [mm]", "V", "sigma_x", "sigma_V"])

x = df["x [mm]"].to_numpy(dtype='float64')
y = df["V"].to_numpy(dtype='float64')

err_x = df["sigma_x"].to_numpy(dtype='float64')
err_y = df["sigma_V"].to_numpy(dtype='float64')

In [46]:
# fit  con solo gli errori su y 
B0 = 1.0     # stima

fit_pre = ROOT.TF1('pre', sinc2, min(x), max(x), 4)
fit_pre.SetParameters(y.max(), B0, x[np.argmax(y)], 0)      # A è Vmax, B è B0, C è posizione del massimo, D è 0
gr0 = ROOT.TGraphErrors(len(x), x, y, np.zeros_like(x), err_y)
gr0.Fit(fit_pre, "SQ0")

par_pre = [fit_pre.GetParameter(i) for i in range(4)]
print(par_pre)

[1.8106679484244224, 0.8079144943994894, 0.14717610358358926, 0.001427855744363808]


In [47]:
#minimizzazzione con ROOT (fit completo)
sinc2Fit = ROOT.TF1('model', sinc2, min(x), max(x), 4)
sinc2Fit.SetParameters(*par_pre)
sinc2Fit.SetParNames("A", "B", "C", "D")

gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Dati sperimentali e fit; x [mm]; V [V]")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)
gr1.SetLineWidth(1)


fit_res = gr1.Fit(sinc2Fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = [sinc2Fit.GetParameter(i) for i in range(4)]
errors = [sinc2Fit.GetParError(i) for i in range(4)]

chi2 = fit_res.Chi2()
ndf = fit_res.Ndf()

err_yeff = np.sqrt(err_y**2 + ((sinc2_deriv(x, par)) * err_x)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"D = {par[3]} ± {errors[3]}")
print(f"Chi2 = {chi2}, ndf = {ndf}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.14, 0.77, 0.42, 0.86)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(sinc2Fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs(f"Immagini/{nome}_fit.png")


Fit status: 0****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      47.7592
NDf                       =           56
Edm                       =  4.98278e-07
NCalls                    =           94
A                         =      1.77148   +/-   0.0089805   
B                         =     0.770489   +/-   0.00434389  
C                         =     0.179365   +/-   0.0260488   
D                         = -3.35136e-05   +/-   0.00115116  

Parametri stimati:
A = 1.7714838564543416 ± 0.008980501292808557
B = 0.7704887081225166 ± 0.004343893254337511
C = 0.17936471910714943 ± 0.02604881247654655
D = -3.351358217721329e-05 ± 0.0011511565137780054
Chi2 = 47.75919369710307, ndf = 56


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file Immagini/rosso_fit.png has been created


In [48]:
if np.abs(par[2]) < 3*errors[2]:
    print("C è compatibile con 0")
else :
    print("C NON è compatibile con 0")

print("---")

if np.abs(par[3]) < 3*errors[3]:
    print("D è compatibile con 0")
else :
    print("D NON è compatibile con 0")

C NON è compatibile con 0
---
D è compatibile con 0


In [49]:
# larghezza della fenditura: d = B*lambda*s/(2 pi)
lam = lam_nm * 1e-6         # lunghezza d'onda in mm
B = abs(par[1])
err_B = errors[1]

d = B * lam * s / (2*np.pi)
err_d = lam/(2*np.pi) * np.sqrt((s*err_B)**2 + (B*err_s)**2)

print(f"d  = {d} ± {err_d} mm")
print(f"2d = {2*d} ± {2*err_d} mm")

d  = 0.199269015553054 ± 0.0013774845611965513 mm
2d = 0.398538031106108 ± 0.0027549691223931025 mm


In [50]:

y_attesi = np.array([sinc2Fit.Eval(x_val) for x_val in x])
residuals_norm = (y - y_attesi) / err_yeff


##NUMERO DI BIN DA USARE (FORMULA BRESSAN)
dev_std = np.std(residuals_norm)
delta_bin = 3.5 * dev_std / (len(residuals_norm))**(1/3)
n_bin= (np.max(residuals_norm) - np.min(residuals_norm))/delta_bin
n_bin = int(np.round(n_bin))
print("numero di bin senza code", n_bin)

##NUMERO DI BIN DA USARE SE PRESENTI CODE (FORMULA BRESSAN)
def IQR(dist):
    return np.percentile(dist, 75) - np.percentile(dist, 25)

delta_bin = 2*IQR(residuals_norm)/ (len(residuals_norm))**(1/3)
n_bin= (np.max(residuals_norm) - np.min(residuals_norm))/delta_bin
n_bin = int(np.round(n_bin))
print("numero di bin con code", n_bin)


gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(residuals_norm), max(residuals_norm), 3)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 10, min(residuals_norm), max(residuals_norm))
for val in residuals_norm:
    hist.Fill(val)

gaussian_fit.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS())

hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors_g = gaussian_fit.GetParErrors()

chi2_g = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors_g[0]}")
print(f"#mu = {parameters[1]} ± {errors_g[1]}")
print(f"#sigma = {parameters[2]} ± {errors_g[2]}")
print(f"Chi2 = {chi2_g}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()

ROOT.gStyle.SetOptFit(1111)
c2.Draw()
c2.SaveAs(f"Immagini/{nome}_hist.png")

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


numero di bin senza code 5
numero di bin con code 6
Fit status: 0
Parametri stimati:
const = 9.55758772847819 ± 1.7771676720750134
#mu = 0.04544391099288267 ± 0.18766390036812222
#sigma = 1.1065881917993552 ± 0.21916893745367133
Chi2 = 2.9899395431276554
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      2.98994
NDf                       =            6
Edm                       =  4.40848e-06
NCalls                    =           69
const                     =      9.55759   +/-   1.77717     
mu                        =    0.0454439   +/-   0.187664    
sigma                     =      1.10659   +/-   0.219169     	 (limited)


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file Immagini/rosso_hist.png has been created


In [51]:
if np.abs(parameters[1]) < 3*errors_g[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")
if np.abs(parameters[2] - 1) < 3*errors_g[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

mu è compatibile con 0
---
sigma è compatibile con 1


In [52]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = y - y_attesi

line_fit = ROOT.TF1('line', line, min(x), max(x), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(x), x, residuals, np.zeros_like(residuals), err_yeff)
gr2.SetTitle("Residui in funzione di x; x [mm]; V_{misurato} − V_{fit} [V]")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors_l = line_fit.GetParErrors()

chi2_l = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors_l[0]}")
print(f"q = {parameters[1]} ± {errors_l[1]}")
print(f"Chi2 = {chi2_l}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs(f"Immagini/{nome}_residui.png")

Fit status: 0
Parametri stimati:
m = 2.1412270941903595e-05 ± 0.00010923182739674828
q = 1.0336375768577916e-06 ± 0.0011144738034813257
Chi2 = 47.72076729231614
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      47.7208
NDf                       =           58
Edm                       =  3.41502e-23
NCalls                    =           42
m                         =  2.14123e-05   +/-   0.000109232 
q                         =  1.03364e-06   +/-   0.00111447  


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file Immagini/rosso_residui.png has been created


## Confronto tra i due laser



In [53]:
d_rosso, err_d_rosso = 0.0, 0.0       
d_verde, err_d_verde = 0.0, 0.0       

sigma_d = abs(d_rosso - d_verde) / np.sqrt(err_d_rosso**2 + err_d_verde**2)
print(f"differenza = {sigma_d:.1f} sigma")

if sigma_d < 3:
    print("d è compatibile tra i due laser")
else :
    print("d NON è compatibile tra i due laser")

differenza = nan sigma
d NON è compatibile tra i due laser


/var/folders/s4/_bthznv52l9_j9tcmc5dxzkh0000gn/T/ipykernel_42575/3981062879.py:4: RuntimeWarning: invalid value encountered in scalar divide
  sigma_d = abs(d_rosso - d_verde) / np.sqrt(err_d_rosso**2 + err_d_verde**2)
